# P03 Разделение обучения и проверки

**Эталон решения**. Полное объяснение находится в Pages. Этот файл содержит исполняемый практикум.

## Цель

Отделить fit от transform и не использовать тестовые данные при подготовке модели.

**Перед началом:** P02: выбор по ошибке, среднее; новое понятие scale вводится в этом примере.

## Подготовка

Запускайте из корня репозитория в его виртуальном окружении. Первичная установка зависимостей описана в README. После установки выполнение не использует сеть, платные API или GPU.

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "mlcourse" / "pilot.py").is_file():
    if ROOT == ROOT.parent:
        raise RuntimeError("Откройте Jupyter из корня ml-llm-course")
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
from mlcourse.pilot import (
    mse, predict_line, linear_gradient, central_difference, fit_standardizer,
    matvec, fit_gradient, make_dataset, run_experiment, predict_payload,
)
plt.rcParams.update({"figure.figsize": (7.2, 4.2), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})
FIGURES = ROOT / "artifacts" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("Данные: искусственные учебные примеры. CPU. Состояние ученика не оценивается автоматически.")

## Разобранный пример

У нормализации тоже есть выученные параметры: среднее и масштаб. Их определяют на train и применяют к новым данным неизменно. Попадание test в fit нарушает независимость проверки, даже если в конкретном маленьком примере метрика случайно не улучшилась.

### Подготовить независимые части

In [ ]:
train_x = np.array([0., 1., 2.]); train_y = np.array([1., 3., 5.])
test_x = np.array([3., 4.]); test_y = np.array([7., 9.])
scaler = fit_standardizer(train_x)
print("mean train:", scaler.mean, "std train:", scaler.scale)
display(pd.DataFrame({"test x": test_x, "transform через train": scaler.transform(test_x)}))
assert scaler.mean == 1
assert np.isclose(scaler.scale, np.sqrt(2/3))

### Показать неправильный fit

In [ ]:
leaky_scaler = fit_standardizer(np.concatenate([train_x, test_x]))
print("Неправильное среднее:", leaky_scaler.mean)
assert leaky_scaler.mean == 2
print("Правильная модель y=2x+1: test MSE =", mse(test_y, predict_line(test_x, 2, 1)))
print("Здесь test используется только для вычисления итоговой ошибки, а не выбора параметров.")

## Практика с поддержкой

Отметьте, какие строки обучают преобразование, какие только применяют его. Почему transform теста не должен менять scaler.mean?

## Самостоятельная задача

Новая задача: train x=[1,2,3], test x=[4,5], зависимость y=2x+1. Найдите среднее только train и ошибочное среднее всех пяти входов. Исправьте fit(concatenate(train,test)) → transform(train) → transform(test). Для центрирования z=x−mean_train выразите y через z. Затем объясните, почему реплики одного собеседования должны попадать в одну часть данных.

Сначала выполните свою попытку. Подсказки и разбор доступны у наставника; использованную помощь отмечайте.

### Эталон

Открывайте после собственной попытки. После разбора понадобится новый вариант для проверки самостоятельности.

In [ ]:
new_train_x = np.array([1., 2., 3.]); new_test_x = np.array([4., 5.])
fixed_scaler = fit_standardizer(new_train_x)
transformed_train = fixed_scaler.transform(new_train_x)
transformed_test = fixed_scaler.transform(new_test_x)
assert fixed_scaler.mean == 2
assert fit_standardizer(np.concatenate([new_train_x, new_test_x])).mean == 3
assert np.isclose(transformed_train.mean(), 0)
np.testing.assert_allclose(transformed_test, np.array([2., 3.]) / np.sqrt(2/3))
z = new_test_x - fixed_scaler.mean
np.testing.assert_allclose(2*z + 5, 2*new_test_x + 1)
print("Среднее train=2; всех данных=3. При z=x−2 получаем y=2z+5. Центрирование и стандартизация — разные преобразования.")
print("Реплики одной сессии распределяем совместно, а не как независимые строки.")

## Проверка

fit получает только train; тест не влияет на параметры или выбор решения; группировка по сессиям объяснена. Критическая ошибка — повторно настроить модель по финальному тесту.

Успешный Run All подтверждает запуск подготовленных примеров. Самостоятельная работа и понимание требуют отдельной проверки наставником.

## Следующий шаг

Временной ряд: можно ли перемешать будущие и прошлые записи? Опишите разделение для прогноза следующей недели.

[Основной источник](https://scikit-learn.org/stable/common_pitfalls.html). Все значения в примерах синтетические. Сохраняйте исходные ноутбуки; для своей работы используйте копию.